# 02 — Data Preprocessing

This notebook extracts hand landmark coordinates from the collected images using **MediaPipe Hands**
and saves them as a structured dataset (`models/data.pickle`) ready for model training.

**Pipeline step:** Raw Images → MediaPipe Landmark Detection → Normalised Features → `data.pickle`

> **Tip:** You can also run this from the terminal:
> ```bash
> python src/preprocess.py
> ```

## 2.1 Imports & Configuration

In [ ]:
import os
import pickle
import sys

import cv2
import matplotlib.pyplot as plt
import mediapipe as mp
import numpy as np

sys.path.insert(0, os.path.abspath('..'))
from src.config import DATA_DIR, LABELS_DICT, MP_DETECTION_CONFIDENCE, PICKLE_PATH

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

print(f'Input  : {DATA_DIR}')
print(f'Output : {PICKLE_PATH}')

## 2.2 Visualise a Sample Image with Landmarks

Before processing the full dataset, let's confirm MediaPipe can detect landmarks on one sample image from each class.

In [ ]:
hands_viz = mp_hands.Hands(static_image_mode=True, min_detection_confidence=MP_DETECTION_CONFIDENCE)

class_dirs = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
fig, axes = plt.subplots(1, len(class_dirs), figsize=(5 * len(class_dirs), 5))

if len(class_dirs) == 1:
    axes = [axes]

for ax, class_dir in zip(axes, class_dirs):
    class_path = os.path.join(DATA_DIR, class_dir)
    sample_img_path = os.path.join(class_path, os.listdir(class_path)[0])
    img = cv2.imread(sample_img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = hands_viz.process(img_rgb)

    annotated = img_rgb.copy()
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(
                annotated, hand_landmarks, mp_hands.HAND_CONNECTIONS,
                mp_drawing_styles.get_default_hand_landmarks_style(),
                mp_drawing_styles.get_default_hand_connections_style()
            )

    label = LABELS_DICT.get(int(class_dir), class_dir)
    ax.imshow(annotated)
    ax.set_title(f'Class {class_dir}: {label}', fontsize=13, fontweight='bold')
    ax.axis('off')

hands_viz.close()
plt.suptitle('MediaPipe Hand Landmarks — Sample Images', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## 2.3 Extract Landmarks from All Images

For each image:
1. Detect hand landmarks with MediaPipe.
2. Normalise `(x, y)` coordinates relative to the hand's bounding box — making features translation-invariant.
3. Collect into `data` and `labels` lists.

In [ ]:
hands = mp_hands.Hands(static_image_mode=True, min_detection_confidence=MP_DETECTION_CONFIDENCE)

data   = []
labels = []
skipped = 0

for class_dir in sorted(class_dirs):
    class_path = os.path.join(DATA_DIR, class_dir)
    images = [f for f in os.listdir(class_path) if f.endswith(('.jpg', '.png'))]
    print(f"Processing class '{class_dir}' ({len(images)} images)...")

    for img_file in images:
        img = cv2.imread(os.path.join(class_path, img_file))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        results = hands.process(img_rgb)

        if not results.multi_hand_landmarks:
            skipped += 1
            continue

        data_aux = []
        for hand_landmarks in results.multi_hand_landmarks:
            x_coords = [lm.x for lm in hand_landmarks.landmark]
            y_coords = [lm.y for lm in hand_landmarks.landmark]
            min_x, min_y = min(x_coords), min(y_coords)
            for lm in hand_landmarks.landmark:
                data_aux.append(lm.x - min_x)
                data_aux.append(lm.y - min_y)

        data.append(data_aux)
        labels.append(class_dir)

hands.close()

print(f'\nExtracted features from {len(data)} images  |  Skipped: {skipped} (no hand detected)')
print(f'Feature vector length: {len(data[0])} values per sample')

## 2.4 Save Preprocessed Dataset

In [ ]:
import os
os.makedirs(os.path.dirname(PICKLE_PATH), exist_ok=True)

with open(PICKLE_PATH, 'wb') as f:
    pickle.dump({'data': data, 'labels': labels}, f)

print(f'Dataset saved → {PICKLE_PATH}')
print(f'Total samples : {len(data)}')
print(f'Classes       : {sorted(set(labels))}')